<a href="https://colab.research.google.com/github/Pritipirojia/AI_Engineering_Tutorial_Notes/blob/main/ML_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## ML Pipeline

### ML Pipeline ---> Pipeline that helps to chain together multiple steps such that output of one step will provided as the input to another step.

Steps of ML Pipeline:
1. Data and separate the data into multiple categories like numerical data, ordinal columns, nominal data
2. Separate transformers pipeline for separate data

*   Numeric data ---> transformer pipeline ---> Imputation(missing data handling) ---> scaling data
*   Ordinal data ---> transformer pipeline ---> Imputation(missing data handling) ---> Encoding (categorical data handling) ---> scaling data
*   Nominal data ---> transformer pipeline ---> Imputation(missing data handling) ---> Encoding (categorical data handling)

3. Cobine these transformers with their respective data.
4. Model Creation
5. Train, test and split data to feed your model & Encoding on your target data(output) if output is categorical

In [1]:
import pandas as pd

In [2]:
df=pd.read_csv('Bank_Marketing_Original_Data.csv')

In [3]:
df.head()

,Age,Job,Marital,Education,Default,Balance,Housing,Loan,contact,Day,Month,Duration,Campaign,Pdays,Previous,poutcome,TARGET
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,261,1,-1,0,unknown,no
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,151,1,-1,0,unknown,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,76,1,-1,0,unknown,no
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,92,1,-1,0,unknown,no
4,33,unknown,single,unknown,no,1,no,no,unknown,5,may,198,1,-1,0,unknown,no


### First step ---> Separate Data into Multiple data categories

In [4]:
df.columns

Index(['Age', 'Job', 'Marital', 'Education', 'Default', 'Balance', 'Housing',
       'Loan', 'contact', 'Day', 'Month', 'Duration', 'Campaign', 'Pdays',
       'Previous', 'poutcome', 'TARGET'],
      dtype='object')

In [6]:
numeric_data=['Age','Balance','Day','Duration','Campaign', 'Pdays',
       'Previous']
ordinal_data=['Education']
nominal_data=['Job', 'Marital', 'Default', 'Balance', 'Housing',
       'Loan', 'contact', 'Month', 'poutcome']

In [7]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.preprocessing import StandardScaler

In [11]:
numeric_pipeline=Pipeline(steps=[('imputer',SimpleImputer(strategy='mean')),('scaling',StandardScaler())])

In [12]:
ordinal_pipeline=Pipeline(steps=[('imputer',SimpleImputer(strategy='most_frequent')),('encoding',OrdinalEncoder(categories=[['unknown','primary','secondary','tertiary']])),('scaling',StandardScaler())])

In [17]:
nominal_pipeline=Pipeline(steps=[('imputer',SimpleImputer(strategy='most_frequent')),('encoding',OneHotEncoder(handle_unknown='ignore'))])

### Third Step ---> Combine Pipeline Transformers with their respective data

In [15]:
from sklearn.compose import ColumnTransformer

In [20]:
features=ColumnTransformer(transformers=[('Numeric Transformation',numeric_pipeline,numeric_data),('Ordinal Transformation',ordinal_pipeline,ordinal_data),('Nominal Transformation',nominal_pipeline,nominal_data)])

In [21]:
features

ColumnTransformer(transformers=[('Numeric Transformation',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('encoding',
                                                  OneHotEncoder(handle_unknown='ignore'))]),
                                 ['Age', 'Balance', 'Day', 'Duration',
                                  'Campaign', 'Pdays', 'Previous']),
                                ('Ordinal Transformation',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('encoding',
                                                  OrdinalEncoder(categories=[['unknown',
                                                                              'primary',
                                                                              'secondary',
                                                                              'tertiary']])),
                                                 ('scaling',
                                                  StandardScaler())]),
                                 ['Education']),
                                ('Nominal Transformation',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('encoding',
                                                  OneHotEncoder(handle_unknown='ignore'))]),
                                 ['Job', 'Marital', 'Default', 'Balance',
                                  'Housing', 'Loan', 'contact', 'Month',
                                  'poutcome'])])

### Fourth Step ---> Create a model with the respective transformers and along with their algorithm

In [22]:
from sklearn.linear_model import LogisticRegression

In [23]:
model = Pipeline(steps=[('features',features),('classifier',LogisticRegression())])

In [25]:
model

Pipeline(steps=[('features',
                 ColumnTransformer(transformers=[('Numeric Transformation',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoding',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['Age', 'Balance', 'Day',
                                                   'Duration', 'Campaign',
                                                   'Pdays', 'Previous']),
                                                 ('Ordinal Transformation',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='m...
                                                                                               'primary',
                                                                                               'secondary',
                                                                                               'tertiary']])),
                                                                  ('scaling',
                                                                   StandardScaler())]),
                                                  ['Education']),
                                                 ('Nominal Transformation',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoding',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['Job', 'Marital', 'Default',
                                                   'Balance', 'Housing', 'Loan',
                                                   'contact', 'Month',
                                                   'poutcome'])])),
                ('classifier', LogisticRegression())])

### Fifth Step ---> Train Test Split data and encoding of output column data

In [26]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

In [27]:
le=LabelEncoder()

In [28]:
x=df.iloc[:,:-1]
y=df['TARGET']

In [30]:
y=le.fit_transform(y)

In [31]:
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2)

### Final step ---> Model training and getting prediction from the model

In [32]:
model.fit(x_train,y_train)

/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Pipeline(steps=[('features',
                 ColumnTransformer(transformers=[('Numeric Transformation',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoding',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['Age', 'Balance', 'Day',
                                                   'Duration', 'Campaign',
                                                   'Pdays', 'Previous']),
                                                 ('Ordinal Transformation',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='m...
                                                                                               'primary',
                                                                                               'secondary',
                                                                                               'tertiary']])),
                                                                  ('scaling',
                                                                   StandardScaler())]),
                                                  ['Education']),
                                                 ('Nominal Transformation',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoding',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['Job', 'Marital', 'Default',
                                                   'Balance', 'Housing', 'Loan',
                                                   'contact', 'Month',
                                                   'poutcome'])])),
                ('classifier', LogisticRegression())])

In [33]:
y_predict=model.predict(x_test)

In [34]:
from sklearn.metrics import accuracy_score

In [35]:
accuracy_score(y_test,y_predict)

0.8971580227800509

In [37]:
from sklearn.ensemble import RandomForestClassifier

In [38]:
model=Pipeline(steps=[('features',features),('classifier',RandomForestClassifier())])

In [39]:
model.fit(x_train,y_train)

Pipeline(steps=[('features',
                 ColumnTransformer(transformers=[('Numeric Transformation',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoding',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['Age', 'Balance', 'Day',
                                                   'Duration', 'Campaign',
                                                   'Pdays', 'Previous']),
                                                 ('Ordinal Transformation',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='m...
                                                                                               'primary',
                                                                                               'secondary',
                                                                                               'tertiary']])),
                                                                  ('scaling',
                                                                   StandardScaler())]),
                                                  ['Education']),
                                                 ('Nominal Transformation',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoding',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['Job', 'Marital', 'Default',
                                                   'Balance', 'Housing', 'Loan',
                                                   'contact', 'Month',
                                                   'poutcome'])])),
                ('classifier', RandomForestClassifier())])

In [40]:
y_predict=model.predict(x_test)

In [41]:
accuracy_score(y_test,y_predict)

0.8950569501271702